# CGC Configuration + Parameter Monitor

Activate a configuration on all 4 PSUs and both switches (normal `swB` + high resolution `swA`),
then run a lightweight housekeeping loop (temperature + device state) every 10 seconds.

Devices: PSU 1-4, swA (SWHR), swB (SW).

## Imports, Logger, Device Instances

In [ ]:
import sys
import os
import time
import logging
from datetime import datetime
from pathlib import Path

# Add src to path
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'src'))

from devices.cgc.psu.psu import PSU
from devices.cgc.sw.sw import SW
from devices.cgc.sw_HR.sw_HR import SWHR

In [ ]:
# Setup external logger
repo_root = Path(os.getcwd()).parent.parent
log_dir = repo_root / "debugging" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = log_dir / f"023_cgc_config_monitor_{timestamp}.log"

logger = logging.getLogger(f"023_cgc_config_monitor_{timestamp}")
logger.setLevel(logging.DEBUG)

file_handler = logging.FileHandler(log_file)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(console_handler)

logger.info("Logger initialized.")
print(f"Log file: {log_file}")

In [ ]:
psu1 = PSU("psu1", com=3, port=0, logger=logger)
psu2 = PSU("psu2", com=4, port=1, logger=logger)
psu3 = PSU("psu3", com=5, port=2, logger=logger)
psu4 = PSU("psu4", com=6, port=3, logger=logger)
swA = SWHR("swA", com=10, stream=0, logger=logger)  # high resolution switch
swB = SW("swB", com=13, port=0, logger=logger)      # normal switch

## Connect All Devices

In [ ]:
psu1.connect()
psu1.set_comspeed(230400)

psu2.connect()
psu2.set_comspeed(230400)

psu3.connect()
psu3.set_comspeed(230400)

psu4.connect()
psu4.set_comspeed(230400)

swA.connect()
swA.set_comspeed(230400)

swB.connect()
swB.set_comspeed(230400)

## Connection Check (read temperature from each device)

In [ ]:
for name, dev in [("psu1", psu1), ("psu2", psu2), ("psu3", psu3), ("psu4", psu4),
                  ("swA", swA), ("swB", swB)]:
    print(f"{name}: {dev.get_sensor_data()}")

## Apply Configuration

Switches first, then PSUs. Set the desired configuration numbers below.

In [ ]:
# --- Switches first ---
swA_config = 0  # TODO: set high-resolution switch config number
swB_config = 0  # TODO: set normal switch config number

# Acquire each device's thread_lock so this is safe to run while the live plot is polling.
with swA.thread_lock:
    swA.load_current_config(swA_config)
with swB.thread_lock:
    swB.load_current_config(swB_config)

In [ ]:
# --- Then PSUs ---
psu1_config = 0  # TODO: set PSU1 config number
psu2_config = 0  # TODO: set PSU2 config number
psu3_config = 0  # TODO: set PSU3 config number
psu4_config = 0  # TODO: set PSU4 config number

# Acquire each device's thread_lock so this is safe to run while the live plot is polling.
with psu1.thread_lock:
    psu1.load_current_config(psu1_config)
with psu2.thread_lock:
    psu2.load_current_config(psu2_config)
with psu3.thread_lock:
    psu3.load_current_config(psu3_config)
with psu4.thread_lock:
    psu4.load_current_config(psu4_config)

## Simplified Housekeeping Loop

Reads temperature (`get_sensor_data`) and device state (`get_device_state`) from every device every 10s.
Stop with KeyboardInterrupt.

In [ ]:
interval = 10  # seconds

devices = [
    ("psu1", psu1),
    ("psu2", psu2),
    ("psu3", psu3),
    ("psu4", psu4),
    ("swA", swA),
    ("swB", swB),
]

def hk_cycle():
    for name, dev in devices:
        try:
            with dev.thread_lock:
                s_status, t0, t1, t2 = dev.get_sensor_data()
                d_status, state_hex, state_names = dev.get_device_state()
            logger.info(
                f"{name} | T0={t0:.1f} T1={t1:.1f} T2={t2:.1f} degC | "
                f"state={state_hex} {state_names}"
            )
        except Exception as e:
            logger.error(f"{name} housekeeping failed: {e}")

logger.info(f"Starting housekeeping loop (interval={interval}s). Ctrl+C to stop.")
try:
    while True:
        hk_cycle()
        time.sleep(interval)
except KeyboardInterrupt:
    logger.info("Housekeeping loop stopped.")

## Live Temperature Plot (web-based, opens in Chrome)

Bokeh server with one subplot per device, 3 temperature sensors each, ~1 Hz update.
Stop with the cell below.

In [ ]:
import threading
import webbrowser
from datetime import timedelta

from bokeh.plotting import figure
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, DatetimeTickFormatter, Range1d
from bokeh.palettes import Category10
from bokeh.server.server import Server
from tornado.ioloop import IOLoop

# Plot settings
POLL_INTERVAL_MS = 1000   # 1 Hz update rate
ROLLING_WINDOW_MIN = 5    # minutes of data visible in plot
MAX_POINTS = 7200         # ~2 hours of data at 1 Hz
BOKEH_PORT = 5007         # different from 020's 5006 in case both run

PLOT_DEVICES = [
    ("psu1", psu1),
    ("psu2", psu2),
    ("psu3", psu3),
    ("psu4", psu4),
    ("swA",  swA),
    ("swB",  swB),
]

# Suppress console logging during live monitoring (file logging continues)
console_handler.setLevel(logging.WARNING)


def make_document(doc):
    """Build the Bokeh document with one subplot per device, 3 sensors each."""
    sources = {
        name: ColumnDataSource(data={"time": [], "t0": [], "t1": [], "t2": []})
        for name, _ in PLOT_DEVICES
    }

    now = datetime.now()
    x_range = Range1d(start=now - timedelta(minutes=ROLLING_WINDOW_MIN), end=now)

    colors = Category10[3]
    figures = []
    for i, (name, _) in enumerate(PLOT_DEVICES):
        p = figure(
            title=name,
            x_axis_type="datetime",
            height=180,
            width=1000,
            x_range=x_range,
            tools="pan,wheel_zoom,box_zoom,reset,save",
            active_scroll="wheel_zoom",
        )
        for sensor_key, sensor_label, color in zip(
            ["t0", "t1", "t2"],
            ["Sensor 0", "Sensor 1", "Sensor 2"],
            colors,
        ):
            p.line(
                "time", sensor_key,
                source=sources[name],
                line_width=2,
                color=color,
                legend_label=sensor_label,
            )
        p.yaxis.axis_label = "Temp [degC]"
        p.xaxis.formatter = DatetimeTickFormatter(
            seconds="%H:%M:%S",
            minsec="%H:%M:%S",
            minutes="%H:%M:%S",
            hourmin="%H:%M:%S",
            hours="%H:%M:%S",
        )
        p.legend.location = "top_left"
        p.legend.click_policy = "hide"
        if i < len(PLOT_DEVICES) - 1:
            p.xaxis.axis_label = ""
        else:
            p.xaxis.axis_label = "Time"
        figures.append(p)

    layout = column(*figures, sizing_mode="stretch_width")
    doc.add_root(layout)
    doc.title = "CGC Temperature Monitor"

    def update():
        try:
            now = datetime.now()
            for name, dev in PLOT_DEVICES:
                try:
                    # Acquire device thread_lock so concurrent kernel-side calls
                    # (e.g. load_current_config) don't race on the same COM port.
                    with dev.thread_lock:
                        status, t0, t1, t2 = dev.get_sensor_data()
                    if status == dev.NO_ERR:
                        sources[name].stream(
                            {"time": [now], "t0": [t0], "t1": [t1], "t2": [t2]},
                            rollover=MAX_POINTS,
                        )
                        logger.info(
                            f"{name} T0={t0:.1f} T1={t1:.1f} T2={t2:.1f} degC"
                        )
                except Exception as e:
                    logger.error(f"{name} sensor read failed: {e}")
            x_range.start = now - timedelta(minutes=ROLLING_WINDOW_MIN)
            x_range.end = now
        except Exception as e:
            logger.error(f"Plot update failed: {e}")

    doc.add_periodic_callback(update, POLL_INTERVAL_MS)


def _start_server():
    io_loop = IOLoop()
    io_loop.make_current()
    server = Server(
        {"/": make_document},
        port=BOKEH_PORT,
        io_loop=io_loop,
        allow_websocket_origin=[f"localhost:{BOKEH_PORT}"],
    )
    server.start()
    io_loop.start()


server_thread = threading.Thread(target=_start_server, daemon=True)
server_thread.start()

time.sleep(1)  # give the server a moment to start
url = f"http://localhost:{BOKEH_PORT}/"
try:
    chrome = webbrowser.get("C:/Program Files/Google/Chrome/Application/chrome.exe %s")
    chrome.open(url)
    print(f"Opened Chrome at {url}")
except webbrowser.Error:
    webbrowser.open(url)
    print(f"Chrome not found, opened default browser at {url}")

print(f"Bokeh server running at {url}")
print("Server runs in background. Run the next cell to restore console logging.")

# Device Commands while devs are running

### Stop Live Plot

In [ ]:
console_handler.setLevel(logging.DEBUG)
logger.info("Live plot logging restored. Bokeh server thread will exit when kernel stops.")
print("Console logging restored. (Browser tab can be closed manually.)")

## Disconnect

In [ ]:
psu1.disconnect()
psu2.disconnect()
psu3.disconnect()
psu4.disconnect()
swA.disconnect()
swB.disconnect()